In [1]:
import unsloth # MUST BE FIRST
from unsloth import FastLanguageModel
import torch
import os
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# --- Configuration ---
model_name = "Qwen/Qwen3.5-4B" # Fixed! (Change to "Qwen/Qwen3.5-4B-Instruct" if you are using the instruct version)
dataset_path = "csv/train.csv"
output_dir = "./qwen-svg-lora"

os.makedirs(output_dir, exist_ok=True)

# --- Hardware Check ---
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: CUDA is not available. Training will fail or be extremely slow.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/rohan/nyu_svg_contest/unsloth_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[unsloth.import_fixes|WARNING]Unsloth: vLLM was built for CUDA 12 but this system has CUDA 13.0. Please reinstall vLLM with the correct CUDA version:

  uv pip install https://github.com/vllm-project/vllm/releases/download/v0.18.0/vllm-0.18.0+cu130-cp38-abi3-manylinux_2_35_x86_64.whl


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.11.0+cu130
GPU Detected: NVIDIA GeForce RTX 3090
Total VRAM: 24.00 GB


In [2]:
import random

# 1. Load the dataset
print(f"Loading dataset from {dataset_path}...")
dataset = load_dataset("csv", data_files=dataset_path, split="train")

# 2. Filter out any broken rows (ensures no null values in prompt or svg)
dataset = dataset.filter(lambda x: x["prompt"] is not None and x["svg"] is not None)

# 3. Define the strict System Prompt for constraints
SYSTEM_PROMPT = """You are an expert SVG coder. You must generate valid, complete SVG code that matches the user's description.
STRICT CONSTRAINTS:
1. Return ONLY the raw SVG code. Do not use markdown code blocks (```svg) and do not provide any explanations.
2. Use ONLY allowed SVG elements (svg, g, path, rect, circle, ellipse, line, polyline, polygon, defs, etc.).
3. Never use <image>, <foreignObject>, or HTML tags.
4. Ensure all tags are properly closed and the XML is strictly well-formed."""

# 4. Define the formatting function for a SINGLE example
def format_single_example(example):
    system_text = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
    user_text = f"<|im_start|>user\nGenerate an SVG image for the following prompt: {example['prompt']}<|im_end|>\n"
    assistant_text = f"<|im_start|>assistant\n{example['svg']}<|im_end|>"
    # We return a dictionary with the new "text" column
    return {"text": system_text + user_text + assistant_text}

# 5. Map the dataset (this actually creates the new column)
print("Pre-formatting dataset into ChatML format...")
dataset = dataset.map(format_single_example, remove_columns=dataset.column_names)

print(f"✅ Prepared {len(dataset)} samples.")
print("\n--- FINAL DATASET STRUCTURE FOR REPORT ---")
# Let's peek at the first row to ensure it looks right
print(dataset[0]["text"][:600] + " ... [SVG TRUNCATED] ... <|im_end|>")

Loading dataset from csv/train.csv...
Pre-formatting dataset into ChatML format...
✅ Prepared 50000 samples.

--- FINAL DATASET STRUCTURE FOR REPORT ---
<|im_start|>system
You are an expert SVG coder. You must generate valid, complete SVG code that matches the user's description.
STRICT CONSTRAINTS:
1. Return ONLY the raw SVG code. Do not use markdown code blocks (```svg) and do not provide any explanations.
2. Use ONLY allowed SVG elements (svg, g, path, rect, circle, ellipse, line, polyline, polygon, defs, etc.).
3. Never use <image>, <foreignObject>, or HTML tags.
4. Ensure all tags are properly closed and the XML is strictly well-formed.<|im_end|>
<|im_start|>user
Generate an SVG image for the following prompt: The image features two orang ... [SVG TRUNCATED] ... <|im_end|>


In [3]:
from unsloth import FastLanguageModel
import torch
import gc

# --- 1. PRE-FLIGHT CLEANUP ---
# Ensure the 3090 is totally empty before we start
if "model" in locals(): del model
if "tokenizer" in locals(): del tokenizer
gc.collect()
torch.cuda.empty_cache()

torch.backends.cuda.matmul.allow_tf32 = True

model_name = "unsloth/Qwen3.5-2B"

print(f"Loading {model_name} and forcing CUDA placement...")

# --- 2. LOAD WITH SDPA ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 1536,
    load_in_4bit = True,
    fast_inference = False,
    # Force the native PyTorch fast-path
    attn_implementation = "sdpa", 
)

# --- 3. APPLY LORA ---
print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. THE SPEED FIXES ---
# Force model to GPU and use bfloat16 for Ampere (3090) architecture
model = model.to("cuda") 
model = model.to(torch.bfloat16)

model.config.use_cache = False 

# Final Check
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")
print(f"Attention: {model.config._attn_implementation}")
print("✅ Model is now HARD-LOADED on GPU. Ready for Cell 4.")

Loading unsloth/Qwen3.5-2B and forcing CUDA placement...
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.3.0. vLLM: 0.18.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 24.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights:   0%|          | 1/617 [00:00<01:54,  5.39it/s]/home/rohan/nyu_svg_contest/unsloth_env/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 617/617 [00:01<00:00, 486.83it/s]


Applying LoRA adapters...
Unsloth: Making `model.base_model.model.model.language_model` require gradients
Device: cuda:0
Dtype: torch.bfloat16
Attention: sdpa
✅ Model is now HARD-LOADED on GPU. Ready for Cell 4.


In [ ]:
from trl import SFTConfig, SFTTrainer
import matplotlib.pyplot as plt
import os
import torch

# --- CONFIGURATION ---
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.cuda.empty_cache()

# 1. Initialize Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1536,
    packing = False,               # KEEP FALSE for speed in your environment
    dataset_num_proc = 1,
    args = SFTConfig(
        per_device_train_batch_size = 4,   
        gradient_accumulation_steps = 4,   
        warmup_steps = 10,
        max_steps = 500,
        learning_rate = 2e-4,
        bf16 = True,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = OUTPUT_DIR,
        
        # --- DATA & SYSTEM FIXES ---
        dataloader_pin_memory = False,
        dataloader_num_workers = 0,
        gradient_checkpointing = True,

        # --- CHECKPOINT SETTINGS ---
        save_strategy = "steps",
        save_steps = 100,          # Saves every 100 steps
        save_total_limit = 2,      # Keeps only the 2 most recent checkpoints (saves disk space)
        report_to = "none",
    ),
)

# 2. Execute Training
print("🚀 Starting Training...")
trainer.train()

# 3. Save Final Weights for Submission
print("\nMerging and saving full model weights...")
model.save_pretrained_merged(
    os.path.join(OUTPUT_DIR, "full_submission_model"), 
    tokenizer, 
    save_method = "merged_16bit"
)

# 4. Generate Loss Graph
print("\nGenerating Training Loss Graph...")
log_history = trainer.state.log_history
steps = [log["step"] for log in log_history if "loss" in log]
losses = [log["loss"] for log in log_history if "loss" in log]

if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, label="Train Loss", color="#3498db", linewidth=2)
    plt.title("SVG Fine-Tuning: Training Loss (SDPA Mode)", fontsize=14)
    plt.xlabel("Steps", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.savefig(os.path.join(OUTPUT_DIR, "loss_chart.png"))
    plt.show()
    print(f"📈 Graph saved as '{OUTPUT_DIR}/loss_chart.png'")

print(f"✅ All Done! Submission weights are in: {OUTPUT_DIR}/full_submission_model")